In [ ]:
# Prerequisites:
# pip install torch torchvision scikit-learn matplotlib pandas numpy

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
)

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from google.colab import drive

In [3]:
drive.mount('/content/drive')
PROJECT_DIR = "/content/drive/MyDrive/ML2 Final Project"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Config

In [5]:
TRAIN_CSV = "train_split.csv"       # Path to your train split
VAL_CSV   = "val_split.csv"         # Path to your val split
TEST_CSV  = "test_split.csv"        # Path to your test split

# Column names in your CSVs
MEL_PATH_COL = "mel_path"           # Column containing path to .npy mel-spectrogram
LABEL_COL    = "label"              # Column containing integer emotion label

NUM_CLASSES  = 7
BATCH_SIZE   = 32
LEARNING_RATE = 1e-3
NUM_EPOCHS   = 50
PATIENCE     = 10                   # Early stopping patience
DEVICE       = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Loading Dataset

**Loading mel-spectrograms from .npy files**



In [6]:
class MelSpectrogramDataset(Dataset):
    """
    Loads mel-spectrogram .npy files and their emotion labels.

    Each .npy file has shape (128, T) where T is the number of time frames.
    We need all spectrograms to be the same width for batching, so we
    pad or truncate to a fixed width (max_len).
    """

    def __init__(self, csv_path, mel_col=MEL_PATH_COL, label_col=LABEL_COL, max_len=130):
        self.df = pd.read_csv(csv_path)
        self.mel_col = mel_col
        self.label_col = label_col
        self.max_len = max_len

        # Drop any rows with missing paths
        self.df = self.df.dropna(subset=[mel_col, label_col]).reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        # Load the mel-spectrogram
        mel = np.load(row[self.mel_col])  # Shape: (128, T)

        # Pad or truncate to fixed width
        if mel.shape[1] < self.max_len:
            pad_width = self.max_len - mel.shape[1]
            mel = np.pad(mel, ((0, 0), (0, pad_width)), mode="constant")
        else:
            mel = mel[:, :self.max_len]

        # Normalize to zero mean, unit variance (per-sample)
        mean = mel.mean()
        std = mel.std()
        if std > 0:
            mel = (mel - mean) / std

        # Add channel dimension: (128, T) → (1, 128, T) for Conv2d
        mel = mel[np.newaxis, :, :]

        # Convert to tensors
        mel_tensor = torch.FloatTensor(mel)
        label_tensor = torch.LongTensor([int(row[self.label_col])])

        return mel_tensor, label_tensor.squeeze()

# Model: Simple CNN Architecture

In [7]:
class EmotionCNN(nn.Module):
    """
    4-block CNN for mel-spectrogram classification.

    Each block: Conv2d → BatchNorm → ReLU → MaxPool → Dropout
    Followed by: Global Average Pooling → FC → Dropout → FC (output)

    Input shape:  (batch, 1, 128, 130)  — 1 channel, 128 mel bands, 130 time frames
    Output shape: (batch, NUM_CLASSES)
    """

    def __init__(self, num_classes=NUM_CLASSES, in_channels=1):
        super(EmotionCNN, self).__init__()

        # Block 1: 1 → 32 filters
        self.block1 = nn.Sequential(
            nn.Conv2d(in_channels, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout(0.3),
        )

        # Block 2: 32 → 64 filters
        self.block2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout(0.3),
        )

        # Block 3: 64 → 128 filters
        self.block3 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout(0.3),
        )

        # Block 4: 128 → 256 filters
        self.block4 = nn.Sequential(
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout(0.3),
        )

        # Global Average Pooling — collapses spatial dims to (batch, 256)
        self.global_pool = nn.AdaptiveAvgPool2d(1)

        # Classifier head
        self.classifier = nn.Sequential(
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.block4(x)
        x = self.global_pool(x)
        x = x.view(x.size(0), -1)  # Flatten: (batch, 256, 1, 1) → (batch, 256)
        x = self.classifier(x)
        return x

# Training Loop

In [8]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    """Train for one epoch, return average loss and predictions."""
    model.train()
    running_loss = 0.0
    all_preds = []
    all_labels = []

    for mel, labels in loader:
        mel, labels = mel.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(mel)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * mel.size(0)
        preds = outputs.argmax(dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    avg_loss = running_loss / len(loader.dataset)
    f1 = f1_score(all_labels, all_preds, average="weighted")
    return avg_loss, f1

In [9]:
def validate(model, loader, criterion, device):
    """Evaluate on validation set, return average loss and predictions."""
    model.eval()
    running_loss = 0.0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for mel, labels in loader:
            mel, labels = mel.to(device), labels.to(device)

            outputs = model(mel)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * mel.size(0)
            preds = outputs.argmax(dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    avg_loss = running_loss / len(loader.dataset)
    f1 = f1_score(all_labels, all_preds, average="weighted")
    return avg_loss, f1, all_preds, all_labels

# Plotting Functions

In [10]:
def plot_training_curves(train_losses, val_losses, train_f1s, val_f1s, save_path=None):
    """Plot training vs validation loss and F1 curves."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    epochs = range(1, len(train_losses) + 1)

    # Loss curves
    ax1.plot(epochs, train_losses, "b-", label="Train Loss", linewidth=2)
    ax1.plot(epochs, val_losses, "r-", label="Val Loss", linewidth=2)
    ax1.set_xlabel("Epoch")
    ax1.set_ylabel("Loss")
    ax1.set_title("Training vs Validation Loss")
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    # F1 curves
    ax2.plot(epochs, train_f1s, "b-", label="Train F1", linewidth=2)
    ax2.plot(epochs, val_f1s, "r-", label="Val F1", linewidth=2)
    ax2.set_xlabel("Epoch")
    ax2.set_ylabel("Weighted F1")
    ax2.set_title("Training vs Validation F1")
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"Saved training curves to {save_path}")
    plt.show()

In [11]:
def plot_confusion_matrix(labels, preds, label_names, save_path=None):
    """Plot a confusion matrix."""
    cm = confusion_matrix(labels, preds)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=label_names)

    fig, ax = plt.subplots(figsize=(8, 8))
    disp.plot(ax=ax, cmap="Blues", values_format="d")
    ax.set_title("Confusion Matrix — CNN Model 1")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"Saved confusion matrix to {save_path}")
    plt.show()

# Run 1-Channel CNN

In [17]:
print(f"Device: {DEVICE}")
print(f"Loading data...")

# ── Create datasets and dataloaders ──
train_dataset = MelSpectrogramDataset(os.path.join(PROJECT_DIR, TRAIN_CSV), max_len=130)
val_dataset   = MelSpectrogramDataset(os.path.join(PROJECT_DIR, VAL_CSV), max_len=130)
test_dataset  = MelSpectrogramDataset(os.path.join(PROJECT_DIR, TEST_CSV), max_len=130)

for dataset in [train_dataset, val_dataset, test_dataset]:
    dataset.df["mel_path"] = dataset.df["mel_path"].apply(
        lambda p: os.path.join(PROJECT_DIR, "mel_spectrograms", p.replace("\\", "/").split("/")[-1])
    )

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f"Train: {len(train_dataset)} clips")
print(f"Val:   {len(val_dataset)} clips")
print(f"Test:  {len(test_dataset)} clips")

# Verify a sample
sample_mel, sample_label = train_dataset[0]
print(f"Sample input shape: {sample_mel.shape}")  # Should be (1, 128, 130)
print(f"Sample label: {sample_label.item()}")

Device: cuda
Loading data...
Train: 3940 clips
Val:   208 clips
Test:  380 clips
Sample input shape: torch.Size([1, 128, 130])
Sample label: 4


In [18]:
# ── Compute class weights to handle imbalance ──
train_labels = train_dataset.df[LABEL_COL].values
class_counts = np.bincount(train_labels.astype(int), minlength=NUM_CLASSES)
class_weights = 1.0 / (class_counts + 1e-6)
class_weights = class_weights / class_weights.sum() * NUM_CLASSES  # Normalize
class_weights_tensor = torch.FloatTensor(class_weights).to(DEVICE)
print(f"Class counts: {class_counts}")
print(f"Class weights: {class_weights.round(3)}")

Class counts: [565 565 565 565 550 565 565]
Class weights: [0.996 0.996 0.996 0.996 1.023 0.996 0.996]


In [20]:
# ── Initialize model, loss, optimizer ──
model = EmotionCNN(num_classes=NUM_CLASSES, in_channels=1).to(DEVICE)
criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
  optimizer, mode="min", patience=5, factor=0.5
)

In [21]:
# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nModel parameters: {total_params:,} total, {trainable_params:,} trainable")
print(f"Architecture:\n{model}\n")


Model parameters: 422,599 total, 422,599 trainable
Architecture:
EmotionCNN(
  (block1): Sequential(
    (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Dropout(p=0.3, inplace=False)
  )
  (block2): Sequential(
    (0): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Dropout(p=0.3, inplace=False)
  )
  (block3): Sequential(
    (0): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, 

In [22]:
# ── Training loop with early stopping ──
train_losses, val_losses = [], []
train_f1s, val_f1s = [], []
best_val_f1 = 0.0
patience_counter = 0
best_model_path = os.path.join("best_cnn_model1.pt")

print("=" * 60)
print(f"{'Epoch':>5} | {'Train Loss':>10} | {'Val Loss':>10} | {'Train F1':>8} | {'Val F1':>8}")
print("=" * 60)

for epoch in range(1, NUM_EPOCHS + 1):
  # Train
  train_loss, train_f1 = train_one_epoch(model, train_loader, criterion, optimizer, DEVICE)

  # Validate
  val_loss, val_f1, _, _ = validate(model, val_loader, criterion, DEVICE)

  # Step the scheduler
  scheduler.step(val_loss)

  # Log
  train_losses.append(train_loss)
  val_losses.append(val_loss)
  train_f1s.append(train_f1)
  val_f1s.append(val_f1)

  print(f"{epoch:>5} | {train_loss:>10.4f} | {val_loss:>10.4f} | {train_f1:>8.4f} | {val_f1:>8.4f}")

  # Early stopping check
  if val_f1 > best_val_f1:
      best_val_f1 = val_f1
      patience_counter = 0
      torch.save(model.state_dict(), best_model_path)
  else:
      patience_counter += 1
      if patience_counter >= PATIENCE:
          print(f"\nEarly stopping at epoch {epoch}. Best val F1: {best_val_f1:.4f}")
          break

print(f"\nTraining complete. Best validation F1: {best_val_f1:.4f}")
print(f"Best model saved to: {best_model_path}")

Epoch | Train Loss |   Val Loss | Train F1 |   Val F1


KeyboardInterrupt: 

In [ ]:
# ── Plot training curves ──
plot_training_curves(
  train_losses, val_losses, train_f1s, val_f1s,
  save_path=os.path.join("model1_training_curves.png"),
)

In [ ]:
# ── Evaluate on test set ──
print("\n" + "=" * 60)
print("FINAL EVALUATION ON TEST SET")
print("=" * 60)

# Load best model
model.load_state_dict(torch.load(best_model_path, weights_only=True))
test_loss, test_f1, test_preds, test_labels = validate(model, test_loader, criterion, DEVICE)

print(f"\nTest Loss: {test_loss:.4f}")
print(f"Test Weighted F1: {test_f1:.4f}")

# Load label names for readable output
label_names = None
if os.path.exists("label_map.json"):
  import json
  with open("label_map.json", "r") as f:
      label_data = json.load(f)
  # Handle both formats: {"angry": 0, ...} or {"label_to_int": {...}, "int_to_label": {...}}
  if "int_to_label" in label_data:
      int_to_label = label_data["int_to_label"]
  else:
      int_to_label = {str(v): k for k, v in label_data.items()}
  label_names = [int_to_label[str(i)] for i in range(NUM_CLASSES)]
else:
  label_names = [str(i) for i in range(NUM_CLASSES)]

In [ ]:
# Classification report
print(f"\nClassification Report:")
print(classification_report(test_labels, test_preds, target_names=label_names))

# Confusion matrix
plot_confusion_matrix(
  test_labels, test_preds, label_names,
  save_path=os.path.join(OUTPUT_DIR, "model1_confusion_matrix.png"),
)

In [ ]:
# ── Save results summary ──
results = {
  "model": "CNN_Model1_FromScratch",
  "input": "mel_spectrogram_1channel",
  "best_val_f1": round(best_val_f1, 4),
  "test_f1": round(test_f1, 4),
  "test_loss": round(test_loss, 4),
  "total_params": total_params,
  "epochs_trained": len(train_losses),
  "best_epoch": len(train_losses) - patience_counter,
}

results_path = os.path.join("1channel_cnn_results.json")
import json
with open(results_path, "w") as f:
  json.dump(results, f, indent=2)
print(f"\nResults saved to {results_path}")
print(json.dumps(results, indent=2))